In [2]:
import kagglehub
import pandas as pd

## Data Download

In [3]:
# Download the consoldiated spam data set from kaggle; the kaggle API handles local caching
path = kagglehub.dataset_download("nitishabharathi/email-spam-dataset")

In [4]:
# Enron spam data set (extracted from emails obtained from the collapse of Enron in 2001 and subsequent
# investigation by the Federal Energy Regulatory Comission, released by Andrew McCallum, and later
# packaged for easier analysis by CMU
df_enron = pd.read_csv(f"{path}/enronSpamSubset.csv")

# Spam assassin data set, collated by Justin Mason between 2002-2003 as a **testing** resource 
# for spam filter developers.
df_spam_assassin = pd.read_csv(f"{path}/completeSpamAssassin.csv")

# LingSpam data set, collated by researcherss at NCSR Demokritos in 2000 from a linguistics mailing list
# as a benchmark for anti-spam filtering
df_lingspam = pd.read_csv(f"{path}/lingSpam.csv")

raw_data_sets = { 
    'Enron': df_enron,
    'Spam Assassin': df_spam_assassin,
    'LingSpam': df_lingspam
}

In [5]:
# First pass analysis of data
def analyze_df(df : pd.DataFrame, label_col: str):
    summary = {}
    feature_cols = df.columns.drop(label_col)

    # Basic shape of the data
    summary["shape"]  = df.shape
    summary["dtypes"] = df.dtypes.to_dict()
    summary["sample"] = df.sample(min(5, len(df)), random_state=42)

    # Label distribution
    summary["label_counts"] = df[label_col].value_counts().to_dict()
    summary["label_mix"] = df[label_col].value_counts(normalize=True).to_dict()

    # Look for nulls
    summary["missing_values"] = df.isnull().sum().to_dict()

    # Look for duplicates
    summary["duplicate_rows"] = int(df.duplicated().sum())
    summary["duplicate_vals"] = {
        col: int(df[col].duplicated().sum()) for col in feature_cols
    }

    return summary

In [6]:
def build_comparison(summaries: dict[str, dict]) -> dict[str, pd.DataFrame]:
    tables = {}

    tables["Overview"] = pd.DataFrame({
        name: { 
            "rows": summary["shape"][0],
            "cols": summary["shape"][1],
            "duplicate_rows": summary["duplicate_rows"]
        }
        for name, summary in summaries.items()
    }).T

    tables["Label Counts"] = pd.DataFrame({
        name: summary["label_counts"] for name, summary in summaries.items()
    }).T

    tables["Label Mix"] = pd.DataFrame({
        name: summary["label_mix"] for name, summary in summaries.items()
    }).T

    tables["Missing Values"] = pd.DataFrame({
        name: summary["missing_values"] for name, summary in summaries.items()
    }).T


    

    return tables

In [9]:
analysis = { name: analyze_df(df, "Label") for name, df in raw_data_sets.items() }

comparison = build_comparison(analysis)
for k in comparison.keys():
    display(comparison[k].style.set_caption(k))

,rows,cols,duplicate_rows
Enron,10000,4,0
Spam Assassin,6046,3,0
LingSpam,2605,3,0


,1,0
Enron,5000,5000
Spam Assassin,1896,4150
LingSpam,433,2172


,1,0
Enron,0.500000,0.500000
Spam Assassin,0.313596,0.686404
LingSpam,0.166219,0.833781


,Unnamed: 0.1,Unnamed: 0,Body,Label
Enron,0.000000,0.000000,0.000000,0.000000
Spam Assassin,nan,0.000000,1.000000,0.000000
LingSpam,nan,0.000000,0.000000,0.000000


In [10]:
# Data in the Body column looks clean; let's validate that the ^Subject: [text]\n pattern is consistent
for name, df in raw_data_sets.items():
    print(f"[{name}] Percentage starting with Subject (case insensitive): {df_enron['Body'].str.match(r"Subject:", case=False).mean()}")
    print(f"[{name}] Percentage starting with Subject (case sensitive):   {df_enron['Body'].str.match(r"Subject:", case=True).mean()}")


[Enron] Percentage starting with Subject (case insensitive): 1.0
[Enron] Percentage starting with Subject (case sensitive):   1.0
[Spam Assassin] Percentage starting with Subject (case insensitive): 1.0
[Spam Assassin] Percentage starting with Subject (case sensitive):   1.0
[LingSpam] Percentage starting with Subject (case insensitive): 1.0
[LingSpam] Percentage starting with Subject (case sensitive):   1.0


In [21]:
def prep_df(df : pd.DataFrame, name: str, body_col: str, subject_col : str = "Subject", label_col : str = "Label") -> pd.DataFrame:   
    # Drop any rows with an empty body column 
    df = df[df[body_col].notna()]

    # Separate the Subject: and the Body:, assuming that the Subject is the first line of the Body
    # newline separated
    parts = df[body_col].str.partition("\n")
    parts.columns = ['header', 'separator', 'body']

    df_prepped = pd.DataFrame()    
    df_prepped[subject_col] = parts["header"].str[len("Subject:"):].str.strip()
    df_prepped[body_col] = parts["body"]
    df_prepped[label_col] = df[label_col]
    df_prepped["Source"] = name
    df_prepped = df_prepped[["Source", subject_col, body_col, label_col]]
    return df_prepped


In [23]:
df_prepped_enron = prep_df(df_enron, "Enron", "Body", "Subject")
(df_prepped_enron["Subject"] == "").mean()

np.float64(0.0091)

## Feature Engineering

In [24]:
# Extract features outside of the structure of which words appear
def engineer_features(df : pd.DataFrame, subject_col : str = "Subject", body_col : str = "Body") -> pd.DataFrame:
    df = df.copy()

    for col, prefix in [(subject_col, "subject"), (body_col, "body")]:
        text = df[col].str

        # Text Length
        length = text.len()
        word_count = text.split().str.len()

        # Letters and captialization ratios
        alpha_count = text.count(r"[A-Za-z]")
        upper_count = text.count(r"[A-Z]")
        upper_ratio = (upper_count / alpha_count).fillna(0)

        # Digit Ratios
        digit_count = text.count(r"\d")
        digit_ratio = (digit_count / length).fillna(0)

        # Special characters
        exclaim_count = text.count("!")
        dollar_count = text.count(r"\$")

        # Flag for embedded urls
        has_http_url = text.contains(r"https?://", regex=True, case=False, na=False)
        has_web_url = text.contains(r"\bwww\.[a-zA-Z0-9]+\.[a-zA-Z]{2,}", regex=True, case=False, na=False)

        df[f"{prefix}_length"]        = length
        df[f"{prefix}_word_count"]    = word_count
        df[f"{prefix}_upper_ratio"]   = upper_ratio
        df[f"{prefix}_digit_ratio"]   = digit_ratio
        df[f"{prefix}_exclaim_count"] = exclaim_count
        df[f"{prefix}_dollar_count"]  = dollar_count
        df[f"{prefix}_has_http_url"]  = has_http_url
        df[f"{prefix}_has_web_url"]   = has_web_url

    df["has_subject"] = df[subject_col].str.len() > 0


    return df

In [26]:
df_enron_featured = engineer_features(df_prepped_enron)

In [27]:
df_enron_featured

,Source,Subject,Body,Label,subject_length,subject_word_count,subject_upper_ratio,subject_digit_ratio,subject_exclaim_count,subject_dollar_count,...,subject_has_web_url,body_length,body_word_count,body_upper_ratio,body_digit_ratio,body_exclaim_count,body_dollar_count,body_has_http_url,body_has_web_url,has_subject
0,Enron,stock promo mover : cwtd,* * * urgent investor trading alert * * *\n w...,1,24,5,0.0,0.000000,0,0,...,False,6176,1092,0.0,0.011982,5,4,False,False,True
1,Enron,are you listed in major search engines ?,submitting your website in search engines may...,1,40,8,0.0,0.000000,0,0,...,False,848,221,0.0,0.000000,0,0,False,False,True
2,Enron,"important information thu , 30 jun 2005 .","subject : important information thu , 30 jun ...",1,41,8,0.0,0.146341,0,0,...,False,728,140,0.0,0.023352,1,0,False,False,True
3,Enron,= ? utf - 8 ? q ? bask your life with ? =,= ? utf - 8 ? q ? individual incremen ? =\n =...,1,41,14,0.0,0.024390,0,0,...,False,492,104,0.0,0.010163,0,0,False,False,True
4,Enron,""" bidstogo "" is places to go , things to do","hello ,\n privacy policy : to\n permanently o...",1,43,11,0.0,0.000000,0,0,...,False,1230,237,0.0,0.000000,0,0,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,Enron,monday 22 nd oct,"louise ,\n do you have half an hour or so fre...",0,16,4,0.0,0.125000,0,0,...,False,128,33,0.0,0.031250,0,0,False,False,True
9996,Enron,missing bloomberg deals,stephanie -\n i believe i ' ve found these tr...,0,23,3,0.0,0.000000,0,0,...,False,7831,1800,0.0,0.087856,0,5,False,False,True
9997,Enron,eops salary survey questionnaire,we will need to establish a deadline . will f...,0,32,4,0.0,0.000000,0,0,...,False,1301,324,0.0,0.019985,0,0,False,False,True
9998,Enron,q 3 comparison,"hi louise ,\n i have a comparison for the fir...",0,14,3,0.0,0.071429,0,0,...,False,255,63,0.0,0.007843,1,0,False,False,True
